In [ ]:
import torch
from occhio import ToyModel
from occhio.autoencoder import TiedLinearRelu
from occhio.distributions import SparseUniform, CorrelatedPairs
from occhio.model_grid import ModelGrid, Axis
from occhio.visualization import plot_embedding, plot_phase_change

In [ ]:
device = "mps"

# Embeddings

In [ ]:
N_FEATURES = 10
N_HIDDEN = 2

In [ ]:
def create_model(params):
    generator = torch.Generator(device=device).manual_seed(199)

    return ToyModel(
        ae=TiedLinearRelu(N_FEATURES, N_HIDDEN, device=device, generator=generator),
        distribution=CorrelatedPairs(
            N_FEATURES,
            density=1 - params["Sparsity"],
            correlation=params["Correlation"],
            device=device,
            generator=generator,
        ),
        importances=torch.tensor([1, 0.5] * int(N_FEATURES / 2))
        * (0.9 ** torch.arange(N_FEATURES)),
        device=device,
    )


grid = ModelGrid(
    create_model,
    axes=[
        Axis(label="Sparsity", values=[0.0, 0.8, 0.9, 0.99]),
        Axis(label="Correlation", values=[0.0, 0.5, 1]),
        # Axis(label="Importance", values=[1, 0.9, 0.7]),
    ],
)

In [ ]:
grid.fit(20000)

In [ ]:
plot_embedding(grid)

## Observations

- Like with the uniform distribution, sparsity leads to superposition.
- There are different regimes:
    - The most important features may get represented individually
    - Some correlated pairs may collapse into one single direction, in that case their norms tend to decrease as well.

In [ ]:
([1, 0, 5] * (N_FEATURES / 2))

In [ ]:
[1, 2] * (int(N_FEATURES / 2))

In [ ]:
(0.9 ** torch.arange(N_FEATURES))

In [ ]:
torch.tensor([1, 0.5] * int(N_FEATURES / 2)) * (0.9 ** torch.arange(N_FEATURES))

# Phase Transitions

In [ ]:
N_FEATURES = 2
N_HIDDEN = 1
EXPERIMENT_SIZE = 21

In [ ]:
def create_model(params):
    generator = torch.Generator(device=device).manual_seed(199)

    return ToyModel(
        ae=TiedLinearRelu(N_FEATURES, N_HIDDEN, device=device, generator=generator),
        distribution=CorrelatedPairs(
            N_FEATURES,
            density=params["Density"],
            correlation=1,
            device=device,
            generator=generator,
        ),
        importances=params["Relative Importance"] ** torch.arange(N_FEATURES),
        device=device,
    )


grid = ModelGrid(
    create_model,
    axes=[
        Axis(
            label="Relative Importance", values=torch.logspace(-1, 1, EXPERIMENT_SIZE)
        ),
        Axis(label="Density", values=torch.logspace(-2, 0, EXPERIMENT_SIZE)),
    ],
)

In [ ]:
grid.fit(n_epochs=15_000)

In [ ]:
plot_phase_change(grid, tracked_feature=1)

# Observations

# Dimensionality